# Sprint E11 walkthrough: one evening of the book, end to end

Purpose: to follow one evening of the daily long/short paper book from
the seed to the email, tracing one name through every step, so that each
number the live page shows can be derived at a whiteboard and audited
against a stored file. Inputs and outputs are labelled at each step.

Nothing here is typed by hand. Every number is read from an artifact, or
recomputed by the library from an artifact and compared against the
stored answer.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "efb").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"

## 0. The evening being traced

The close is **read from the stored price panel**, so the notebook cannot
claim a date its inputs do not price. The name followed through every step
and the name the variance cap acts on are found in the book, not written
into this cell: `TRACE` is asserted to be present, and the cap case is
whichever name the pre-cap variance shares put highest.

In [2]:
prices = pd.read_parquet(DATA / 'raw' / 'prices.parquet')
CLOSE = pd.Timestamp(prices.index.get_level_values('date').max())
STAMP = str(CLOSE.date())
print('INPUT  data/raw/prices.parquet, the date index, newest session')
print('one evening traced end to end at the close:', STAMP)

INPUT  data/raw/prices.parquet, the date index, newest session
one evening traced end to end at the close: 2026-09-21


In [3]:
# The evening itself, once: `build_proposal` is the whole sizing night, and
# every later cell reproduces one step of it and compares to this manifest.
# Nothing is stored (store=False), so the notebook cannot move the live book.
from live import evening_job as ej

PROPOSAL = ej.build_proposal(as_of=CLOSE, store=False)
NAV = PROPOSAL['nav']
print('OUTPUT the evening manifest, live/evening_job.py::build_proposal')
for field in ('as_of', 'n_names', 'n_excluded', 'n_kept', 'gross', 'net',
              'kept_gross', 'constraint', 'achieved_annual_vol',
              'target_annual_vol', 'gross_cap_bound', 'idio_share_after_fmp',
              'max_abs_exposure_after_fmp', 'expected_establishment_cost_bps',
              'max_input_staleness_days'):
    if field in PROPOSAL:
        print(f'  {field} = {PROPOSAL[field]}')
print('nav (the paper notional) =', NAV)

OUTPUT the evening manifest, live/evening_job.py::build_proposal
  as_of = 2026-09-21
  n_names = 499
  n_excluded = 4
  n_kept = 169
  gross = 0.9685376726947534
  net = -9.992007221626409e-16
  kept_gross = 1.0
  achieved_annual_vol = 0.1
  target_annual_vol = 0.1
  gross_cap_bound = False
  idio_share_after_fmp = 1.0
  max_abs_exposure_after_fmp = 3.819861094100929e-15
  expected_establishment_cost_bps = 15.020392058470854
  max_input_staleness_days = 10
nav (the paper notional) = 1000000.0


## 1. The seed, the appended sessions, and the staleness gate per input

Two universes meet here and the seam is never bridged. History is frozen on
the pinned Wikipedia table through the seed cutoff; the live universe comes
from the SPY archive from 2026-09-18 forward. The model is extended one
session at a time, and the gate then checks **every input** for distance
from the close being priced, in NYSE sessions, against the allowance each
input carries.

In [4]:
from live import appendix, extend, staleness

returns = pd.read_parquet(DATA / 'processed' / 'returns.parquet', columns=['r'])
sessions = pd.DatetimeIndex(sorted(returns.index.get_level_values('date').unique()))
appended = sessions[sessions > appendix.SEED_CUTOFF]
print('INPUT  data/processed/returns.parquet, the date index')
print('  seed cutoff (live/appendix.py::SEED_CUTOFF):', appendix.SEED_CUTOFF.date())
print('  sessions stored in the tree:', len(sessions),
      'from', str(sessions.min().date()), 'to', str(sessions.max().date()))
print('  appended after the cutoff, one fit each:', len(appended))
print('  newest appended:', [str(day.date()) for day in appended[-4:]])
print('  last priced session (live/extend.py):', extend.last_price_session(DATA).date())

INPUT  data/processed/returns.parquet, the date index
  seed cutoff (live/appendix.py::SEED_CUTOFF): 2026-09-03
  sessions stored in the tree: 4204 from 2010-01-04 to 2026-09-21
  appended after the cutoff, one fit each: 11
  newest appended: ['2026-09-16', '2026-09-17', '2026-09-18', '2026-09-21']
  last priced session (live/extend.py): 2026-09-21


In [5]:
# The gate, per input. `sessions_behind` is NYSE sessions and is the number
# that decides; `max_input_staleness_days` is calendar days and is only what
# the email quotes. The universe is the one input allowed to be a session
# behind, because an index snapshot is filed after the close it describes.
GATE = staleness.check(root=DATA, target=CLOSE)
gate_rows = pd.DataFrame([{'input': name, **entry}
                         for name, entry in GATE['inputs'].items()])
print('OUTPUT live/staleness.py::check')
print(gate_rows[['input', 'gated_by', 'content', 'content_sessions_behind',
                 'fetch', 'allowed_sessions_behind', 'sessions_behind']]
      .to_string(index=False))
print('status:', GATE['status'], '| worst input:', GATE['worst_input'],
      '| worst sessions behind:', GATE['worst_sessions_behind'])
print('max_input_staleness_days (calendar):', GATE['max_input_staleness_days'])

OUTPUT live/staleness.py::check
           input gated_by    content  content_sessions_behind      fetch  allowed_sessions_behind  sessions_behind
          prices  content 2026-09-21                        0 2026-09-21                        0                0
     descriptors  content 2026-09-21                        0 2026-09-21                        0                0
  factor_returns  content 2026-09-21                        0 2026-09-21                        0                0
specific_returns  content 2026-09-21                        0 2026-09-21                        0                0
      factor_cov  content 2026-09-21                        0 2026-09-21                        0                0
    specific_var  content 2026-09-21                        0 2026-09-21                        0                0
        universe  content 2026-09-21                        0 2026-09-21                        1                0
          shares    fetch 2026-09-22            

In [6]:
# The universe is asserted, not assumed: it is the newest SPY archive dated
# on or before the close, and a file before the seam is refused.
universe, spy_path = ej.load_spy_universe(DATA, as_of=CLOSE)
SPY = sorted(universe['ticker'].astype(str).str.upper().str.strip())
wide, _counts = ej.eval_risk.load_clean_wide(DATA)
model_names = {str(column) for column in wide.columns}
NAMES = [ticker for ticker in SPY if ticker in model_names]
EXCLUDED = [ticker for ticker in SPY if ticker not in model_names]
print('INPUT  ' + str(spy_path.relative_to(ROOT)))
print('  SPY members', len(SPY), '| known to the frozen model', len(NAMES),
      '| excluded and counted, never imputed', len(EXCLUDED))
print('  excluded:', EXCLUDED)
assert len(NAMES) == PROPOSAL['n_names'], 'the universe moved under the trace'

INPUT  data/raw/spy_holdings/spy_holdings_2026-09-21.parquet
  SPY members 503 | known to the frozen model 499 | excluded and counted, never imputed 4
  excluded: ['BE', 'DD', 'ILMN', 'P']


## 2. The alpha, the specific volatility, and the sized weight

INPUT: `models/XS-v1/specific_returns.parquet` for the signal,
`alpha/summary.parquet` for the IC, `models/XS-v1/specific_var.parquet` (via
the model pieces) for the variance. OUTPUT: the alpha vector the sizing reads.

`signal` is the compounded specific return over t-252 to t-21. It is
standardized across the cross-section, then multiplied by the stored IC, the
specific variance and the shrinkage `kappa`.

In [7]:
from efb import alpha as alpha_mod
from efb import size

PIECES = ej.eval_risk._xs_pieces(CLOSE, NAMES, DATA)
DESIGN, FACTOR_COV, SPECIFIC = PIECES['design'], PIECES['factor_covariance'], PIECES['specific']
IC = ej._stored_ic(ej.SIGNAL, DATA)
KAPPA = alpha_mod.KAPPA
signal = alpha_mod.idio_momentum(DATA)
raw = (signal.loc[pd.to_datetime(signal['date']) == CLOSE]
       .set_index('ticker')['signal'].reindex(NAMES))
Z = ((raw - raw.mean()) / raw.std(ddof=1)).fillna(0.0)
ALPHA = np.where(np.isfinite(SPECIFIC), IC * SPECIFIC * Z * KAPPA,
                 IC * float(np.nanmedian(SPECIFIC)) * Z * KAPPA)
ALPHA = pd.Series(ALPHA, index=NAMES)
print('INPUT  ic =', IC, 'from data/alpha/summary.parquet, ic_h1_mean')
print('INPUT  kappa =', KAPPA, 'from efb/alpha.py::KAPPA')
print('INPUT  the design is', DESIGN.shape, '= names x estimated factors')
SIZED_FULL = size.proportional(ALPHA.to_numpy(float), SPECIFIC)
print()
for ticker in ('LITE', 'MRNA'):
    if ticker not in NAMES:
        print(f'{ticker}: not in the universe at this close')
        continue
    position = NAMES.index(ticker)
    print(f'{ticker}: signal {raw[ticker]:+.6f} | z {Z[ticker]:+.6f} | '
          f'specific variance {SPECIFIC[position]:.6e} | alpha {ALPHA[ticker]:.6e} | '
          f'alpha/specific {SIZED_FULL[position]:+.6f}')

INPUT  ic = 0.01210198660523347 from data/alpha/summary.parquet, ic_h1_mean
INPUT  kappa = 0.1 from efb/alpha.py::KAPPA
INPUT  the design is (499, 17) = names x estimated factors

LITE: signal +2.253120 | z +6.319079 | specific variance 2.247966e-03 | alpha 1.719096e-05 | alpha/specific +0.007647
MRNA: signal +2.416794 | z +6.780151 | specific variance 3.629054e-02 | alpha 2.977759e-04 | alpha/specific +0.008205


In [8]:
# The alpha arithmetic, checked against the stored proposal for the names the
# two books share, so the check is against a stored number and not against
# the notebook's own output.
STORED_BOOK = pd.read_parquet(ROOT / 'live' / 'proposals' / f'proposal_{STAMP}.parquet')
STORED = STORED_BOOK.set_index('ticker')
shared = [ticker for ticker in STORED.index if ticker in ALPHA.index]
gap = float((STORED.loc[shared, 'alpha'] - ALPHA.reindex(shared)).abs().max())
print(f'  alpha recomputed vs stored, on the {len(shared)} names the stored book and')
print(f'  this universe share: worst gap {gap:.3e}')
assert gap < 1e-12, 'the recomputed alpha is not the stored alpha'
print(f'  the stored book for this close: {len(STORED_BOOK)} names, gross '
      f'{STORED_BOOK["weight"].abs().sum():.6f}, net {STORED_BOOK["weight"].sum():+.2e}')

  alpha recomputed vs stored, on the 150 names the stored book and
  this universe share: worst gap 0.000e+00
  the stored book for this close: 150 names, gross 1.000000, net -2.78e-16


## 3. The 20-share floor, iterated to a fixed point

The construction is the owner's: **share only**, a minimum of 20 whole
shares per name, no dollar floor, and enforced on the **final** weights,
which are the ones that trade. The rule is drop-then-admit: drop every name
that ends below its floor, then admit names back in the `|w| / floor` order
while the enlarged set's own final weights still clear every kept name's
floor, and repeat the admission pass until a pass changes nothing.

The reason it iterates: the floor is checked on the vector **after** the
hedge and the renormalization, and every one of those steps rescales the
book. A name is not below its floor because of its own weight alone, so the
check has to be re-run on the re-sized book, not on the raw target.

The rule changed after the stored proposal for this close was written, so the
book below is **not** the stored file. Both are shown, and the traced name is
taken from the book the loop runs today.


In [9]:
from efb import registry

CONSTRUCTION = registry.live_construction(
    registry.load(DATA / 'models' / 'registry.json'), ej.MODEL_VERSION)
print('INPUT  data/models/registry.json, models.XS-v1.live:', CONSTRUCTION)

FULL = size.procedure_6_3(ALPHA.to_numpy(float), DESIGN, FACTOR_COV, SPECIFIC)
FULL, GROSS_CAP_BOUND = ej._scale_to_target(FULL, DESIGN, FACTOR_COV, SPECIFIC)
close = ej._close_prices(CLOSE, DATA)
KEEP, FINALIZE, SEARCH = ej.enforce_floor_by_drop_then_admit(
    NAMES, ALPHA.to_numpy(float), DESIGN, FACTOR_COV, SPECIFIC, close, NAV,
    FULL, dollar_floor=CONSTRUCTION['dollar_floor'],
    share_floor=CONSTRUCTION['share_floor'])
print()
print('OUTPUT live/evening_job.py::enforce_floor_by_drop_then_admit')
print('  drop only passes, floor satisfied:', SEARCH['n_drop_only'])
print('  after one admission pass:', SEARCH['n_one_pass_admission'])
print('  admitted by the passes:', SEARCH['admitted'])
print('  admission passes run:', SEARCH['admit_passes'],
      '| cycles:', SEARCH['cycles'], '| converged:', SEARCH['converged'])
print('  kept:', int(KEEP.sum()), 'of', len(NAMES))

INPUT  data/models/registry.json, models.XS-v1.live: {'construction': 'share_only', 'share_floor': 20, 'dollar_floor': 0.0, 'floor_iterated': True}

OUTPUT live/evening_job.py::enforce_floor_by_drop_then_admit
  drop only passes, floor satisfied: 120
  after one admission pass: 158
  admitted by the passes: 49
  admission passes run: 4 | cycles: 1 | converged: True
  kept: 169 of 499


In [10]:
# The traced name is taken from the book, not typed: the brief names LITE, and
# if the current rule did not keep it the notebook says so rather than
# tracing a name that is not there.
IDX = np.asarray(FINALIZE['idx'], dtype=int)
KEPT_NAMES = [str(name) for name in FINALIZE['names_sub']]
W_RAW = FINALIZE['w_sub']
WANTED = 'LITE'
TRACE = WANTED if WANTED in KEPT_NAMES else KEPT_NAMES[int(np.argmax(np.abs(W_RAW)))]
print('  this close, under the rule the live loop runs:')
print('   dropped by the floor check alone:', SEARCH['n_drop_only'])
print('   kept after one admission pass   :', SEARCH['n_one_pass_admission'])
print('   kept at the fixed point         :', int(KEEP.sum()))
print('   admission passes, cycles        :', SEARCH['admit_passes'], SEARCH['cycles'],
      '| converged:', SEARCH['converged'])
print()
if WANTED in KEPT_NAMES:
    print(f'  the traced name is {TRACE}, as the brief asks')
else:
    print(f'  {WANTED} is NOT in the book the current rule keeps at this close.')
    print(f'  It was in the stored proposal for this close, which was written by the')
    print(f'  earlier floor rule; the drop-then-admit fixed point drops it. The')
    print(f'  trace follows {TRACE} instead, the largest weight that is actually here.')

  this close, under the rule the live loop runs:
   dropped by the floor check alone: 120
   kept after one admission pass   : 158
   kept at the fixed point         : 169
   admission passes, cycles        : 4 1 | converged: True

  LITE is NOT in the book the current rule keeps at this close.
  It was in the stored proposal for this close, which was written by the
  earlier floor rule; the drop-then-admit fixed point drops it. The
  trace follows TER instead, the largest weight that is actually here.


In [11]:
# The stored proposal for this close, against the rule the loop runs today.
# This is the reproducibility caveat the notebook exists to make visible: a
# re-run of an old evening does not reproduce its stored file, because the
# floor rule changed after it was written.
stored_path = ROOT / 'live' / 'proposals' / f'proposal_{STAMP}.json'
if stored_path.exists():
    stored_manifest = json.loads(stored_path.read_text())
    print('  STORED manifest for this close', {
        key: stored_manifest.get(key) for key in
        ('n_names', 'n_kept', 'n_kept_drop_only', 'n_kept_one_pass_admission',
         'floor_search_passes', 'floor_rule', 'gross')})
    print('  RECOMPUTED today         ', {
        'n_names': len(NAMES), 'n_kept': int(KEEP.sum()),
        'n_kept_drop_only': SEARCH['n_drop_only'],
        'n_kept_one_pass_admission': SEARCH['n_one_pass_admission'],
        'floor_search_passes': SEARCH['admit_passes'], 'floor_rule': 'drop_then_admit',
        'gross': round(float(PROPOSAL['gross']), 6)})
    stored_names = set(STORED_BOOK['ticker'])
    now_names = set(KEPT_NAMES)
    print(f'  names in the stored book and not today: {len(stored_names - now_names)}')
    print(f'  names today and not in the stored book: {len(now_names - stored_names)}')
    print('  the rule the loop runs is the one traced; the stored file is the')
    print('  earlier rule, and the two are reported apart rather than averaged.')

  STORED manifest for this close {'n_names': 499, 'n_kept': 150, 'n_kept_drop_only': 119, 'n_kept_one_pass_admission': 143, 'floor_search_passes': 4, 'floor_rule': 'drop_then_admit', 'gross': 0.9685376726947534}
  RECOMPUTED today          {'n_names': 499, 'n_kept': 169, 'n_kept_drop_only': 120, 'n_kept_one_pass_admission': 158, 'floor_search_passes': 4, 'floor_rule': 'drop_then_admit', 'gross': 0.968538}
  names in the stored book and not today: 21
  names today and not in the stored book: 40
  the rule the loop runs is the one traced; the stored file is the
  earlier rule, and the two are reported apart rather than averaged.


## 4. The 10% variance-share cap

No name may carry more than 10% of the book's predicted **specific**
variance, which is the part of the risk no factor hedge removes. The share
is `w_i^2 s_i^2 / sum(w_j^2 s_j^2)`, scale invariant.

It is solved, not iterated. Rescaling a violator changes the total and
re-violates the names just clamped, so the sequence crawls: measured at 100
passes and still moving. The closed form is water-filling, find the level
`L` with `sum(min(c_i, L)) = L / cap`, and every clamped name then sits at
exactly the cap by construction.

It is applied **before** the hedge, and only before, because the hedge is a
projection of the sized vector.

Two books are shown. The first is the book the current rule keeps, which is the
one being traced; the second is the **stored** proposal for this close, where
the brief's own case, MRNA, is the name the cap acts on. Both are the same
arithmetic on different kept sets.

In [12]:
from live import sizing

SPECIFIC_KEPT = SPECIFIC[IDX]
SIZED = size.proportional(ALPHA.to_numpy(float)[IDX], SPECIFIC_KEPT)
SHARES_BEFORE = sizing.variance_shares(SIZED, SPECIFIC_KEPT)
CAPPED = sizing.cap_variance_shares(SIZED, SPECIFIC_KEPT)
# The hedge acts on the renormalized capped book, not the raw capped one: the
# sizing step renormalizes to gross 1.0 before it hedges, so the pre-hedge
# exposure the hedge computes is the exposure of this vector.
CAPPED_RENORM = sizing.renormalize(CAPPED, gross=1.0)
SHARES_AFTER = sizing.variance_shares(CAPPED, SPECIFIC_KEPT)
work = pd.DataFrame({'share_before': SHARES_BEFORE, 'share_after': SHARES_AFTER,
                    'scale': np.divide(CAPPED, SIZED, out=np.ones_like(SIZED),
                                         where=SIZED != 0)},
                   index=KEPT_NAMES)
CAP_CASE = str(work['share_before'].idxmax())
above = work.loc[work['share_before'] > sizing.VARIANCE_SHARE_CAP + 1e-12]
print('INPUT  live/sizing.py::VARIANCE_SHARE_CAP =', sizing.VARIANCE_SHARE_CAP)
print('  names above the cap before capping:', len(above))
print(work.sort_values('share_before', ascending=False).head(5).to_string())
row = work.loc[CAP_CASE]
print()
print(f'the worked case, {CAP_CASE}:')
print(f'  before: {row["share_before"]:.6f} of the book variance in one name')
print(f'  after : {row["share_after"]:.6f}, the cap exactly')
print(f'  its own weight is scaled by {row["scale"]:.6f}, and only downwards')
assert row['share_before'] > sizing.VARIANCE_SHARE_CAP
assert abs(row['share_after'] - sizing.VARIANCE_SHARE_CAP) < 1e-9

INPUT  live/sizing.py::VARIANCE_SHARE_CAP = 0.1
  names above the cap before capping: 4
      share_before  share_after     scale
MU        0.256905          0.1  0.443882
MRVL      0.152941          0.1  0.575296
DELL      0.150753          0.1  0.579456
WDC       0.128794          0.1  0.626912
TER       0.057515          0.1  0.938130

the worked case, MU:
  before: 0.256905 of the book variance in one name
  after : 0.100000, the cap exactly
  its own weight is scaled by 0.443882, and only downwards


In [13]:
# The brief's own worked case, MRNA, lives in the STORED book for this close,
# not in the book the current rule keeps. The arithmetic is the same, so it is
# repeated here on that book and labelled for what it is.
old_names = [str(name) for name in STORED_BOOK['ticker']]
slot = {name: position for position, name in enumerate(NAMES)}
old_rows = [slot[name] for name in old_names if name in slot]
old_sized = size.proportional(ALPHA.to_numpy(float)[old_rows], SPECIFIC[old_rows])
old_before = sizing.variance_shares(old_sized, SPECIFIC[old_rows])
old_capped = sizing.cap_variance_shares(old_sized, SPECIFIC[old_rows])
old_after = sizing.variance_shares(old_capped, SPECIFIC[old_rows])
old_work = pd.DataFrame({'share_before': old_before, 'share_after': old_after},
                       index=[name for name in old_names if name in slot])
print('the stored book for this close, the brief\'s case:')
print(old_work.sort_values('share_before', ascending=False).head(4).to_string())
if 'MRNA' in old_work.index:
    print(f'  MRNA: {old_work.loc["MRNA", "share_before"]:.6f} before, '          f'{old_work.loc["MRNA", "share_after"]:.6f} after the cap')

the stored book for this close, the brief's case:
      share_before  share_after
MRNA      0.843694          0.1
LITE      0.045395          0.1
MU        0.027965          0.1
MRVL      0.016648          0.1
  MRNA: 0.843694 before, 0.100000 after the cap


## 5. The FMP hedge: exposures before and after, and the traced name

The hedge is the exact in-model FMP. It solves `X'X l = X'w` and subtracts
`X l`, so the post-hedge exposure `X'(w - Xl)` is **zero by construction**,
not small. It is exact because the mimicking portfolios are the model's own:
the columns of `X (X'X)^-1`.

The `before` vector is the hedge's own `X'w`, returned by the sizing step
rather than reconstructed, so the page and the hedge cannot disagree.

In [14]:
W_KEPT, PRE_HEDGE = sizing.procedure_6_3_hedged_with_exposures(
    ALPHA.to_numpy(float)[IDX], DESIGN[IDX], FACTOR_COV, SPECIFIC_KEPT,
    sized=CAPPED_RENORM)
W_KEPT = sizing.renormalize(W_KEPT, gross=1.0)
DESIGN_KEPT = DESIGN[IDX]
POST_HEDGE = DESIGN_KEPT.T @ W_KEPT
from efb.models import fundamental as fx
hedge_table = pd.DataFrame({'factor': fx.ESTIMATED_NAMES,
                            'before': PRE_HEDGE, 'after': POST_HEDGE})
hedge_table['moved'] = hedge_table['before'] - hedge_table['after']
print('INPUT  data/models/XS-v1/descriptors.parquet through efb/race.py::_descriptor_design')
print(hedge_table.to_string(index=False))
print()
print('worst exposure after the hedge:', float(np.abs(POST_HEDGE).max()))
print('worst exposure the page shows   :', PROPOSAL['max_abs_exposure_after_fmp'])
assert float(np.abs(POST_HEDGE).max()) < 1e-12

INPUT  data/models/XS-v1/descriptors.parquet through efb/race.py::_descriptor_design
   factor    before         after     moved
   market -0.137339 -8.881784e-16 -0.137339
     size  0.737781 -6.036838e-15  0.737781
     beta  0.344132 -3.469447e-16  0.344132
 momentum  0.969772 -7.042977e-16  0.969772
 reversal  0.413121 -5.273559e-16  0.413121
resid_vol  0.009538 -1.873501e-16  0.009538
liquidity  0.594766  2.414735e-15  0.594766
sector_10 -0.032686  7.806256e-18 -0.032686
sector_15 -0.013795 -3.686287e-17 -0.013795
sector_20 -0.053106  3.209238e-17 -0.053106
sector_25 -0.099338  6.938894e-18 -0.099338
sector_30 -0.007741 -2.038300e-17 -0.007741
sector_35 -0.026216  4.163336e-17 -0.026216
sector_40  0.012028 -8.673617e-17  0.012028
sector_45  0.139379 -5.898060e-17  0.139379
sector_50 -0.026394  6.071532e-18 -0.026394
sector_55  0.009329  8.803722e-17  0.009329

worst exposure after the hedge: 6.036837696399289e-15
worst exposure the page shows   : 3.819861094100929e-15


In [15]:
# The traced name, through the hedge. Its pre-hedge weight is what the cap
# left; the hedge moves it by whatever the mimicking portfolios need at that
# name, which is not small and is not noise.
trace_at = KEPT_NAMES.index(TRACE)
before_w = CAPPED_RENORM[trace_at]
after_w = W_KEPT[trace_at]
print(f'{TRACE} through the hedge:')
print(f'  after the variance cap and the renormalization: {before_w:+.6f}')
print(f'  as the hedge leaves it                          : {after_w:+.6f}')
print(f'  the hedge adjustment                            : {after_w - before_w:+.6f}'
      f'  ({(after_w - before_w) / abs(before_w):+.1%} of its own weight)')
own = pd.Series(before_w * DESIGN_KEPT[trace_at], index=fx.ESTIMATED_NAMES)
print(f'  its own contribution to the book exposure, at {TRACE}, the largest four:')
print(own.reindex(own.abs().sort_values(ascending=False).index).head(4).round(6)
      .to_string())
print('  after the hedge the book exposure is zero for every factor, so the')
print('  adjustment is what the hedge had to do at this one name to get there.')

TER through the hedge:
  after the variance cap and the renormalization: +0.032605
  as the hedge leaves it                          : +0.031628
  the hedge adjustment                            : -0.000977  (-3.0% of its own weight)
  its own contribution to the book exposure, at TER, the largest four:
momentum    0.055355
beta        0.048131
size       -0.042310
market      0.032605
  after the hedge the book exposure is zero for every factor, so the
  adjustment is what the hedge had to do at this one name to get there.


## 6. Renormalization, whole shares, and the orders

Renormalization to gross 1.0 is a single positive scalar on the hedged
vector, which is why it cannot undo the hedge or the cap. Quantization
then truncates to whole shares: `int(|w| NAV / price)`. Truncation, not
rounding, at every step, and the rounding error is reported rather than
absorbed.

In [16]:
gross_before = float(np.abs(W_KEPT).sum())
SHARES = ej.kept_shares(W_KEPT, FINALIZE['prices'], NAV)
QUANT = FINALIZE['quant']
print('  gross before the final renormalization:', PROPOSAL['kept_gross_before_renorm'])
print('  gross after                             :', round(gross_before, 12))
print('  quantization: nav', QUANT['nav'], '| names with a whole-share count:',
      int((SHARES > 0).sum()), '| rounds to zero:', QUANT['long_targets_rounding_to_zero'])
print('  gross weight error as a share of NAV:', QUANT['gross_weight_error'])
dist = QUANT['distribution']
print('  rounding error, pct of target:',
      {k: round(v, 6) for k, v in dist['rounding_error_pct_of_target_quantiles'].items()})

  gross before the final renormalization: 0.4768434181561131
  gross after                             : 1.0
  quantization: nav 1000000.0 | names with a whole-share count: 169 | rounds to zero: 0
  gross weight error as a share of NAV: 0.008033239507675137
  rounding error, pct of target: {'0.0': 8.1e-05, '0.25': 0.003339, '0.5': 0.007281, '0.75': 0.013733, '0.9': 0.021739, '0.99': 0.038392, '1.0': 0.041035}


In [17]:
# The orders. Read from the stored execution log for this close, then the
# client order id recomputed from its own ingredients: date, ticker, side.
# The id is the idempotency ticket, so a re-fire of the same leg collides at
from live import alpaca

log_path = ROOT / 'live' / 'logs' / f'execution_{STAMP}.parquet'
if log_path.exists():
    orders_log = pd.read_parquet(log_path)
    orders_log['client_order_id'] = [
        alpaca.client_order_id(STAMP, str(row.ticker),
                              'buy' if row.intended_notional >= 0 else 'sell')
        for row in orders_log.itertuples(index=False)]
    print('OUTPUT', log_path.relative_to(ROOT), '| rows', len(orders_log))
    print(orders_log.head(5).to_string(index=False))
    print('  statuses:', orders_log['status'].value_counts().to_dict())
else:
    print('no execution log stored for', STAMP)

OUTPUT live/logs/execution_2026-09-21.parquet | rows 27
ticker  intended_notional  filled_notional  status                 reason trade_date                    client_order_id
   ALB       -8389.391207              0.0 DRY_RUN dry run: no order sent 2026-09-21  efb-2026-09-21-ALB-S-d6cb5bc9b5a5
  AMAT        7737.097478              0.0 DRY_RUN dry run: no order sent 2026-09-21 efb-2026-09-21-AMAT-B-b2699dae572b
   APH       -5990.041755              0.0 DRY_RUN dry run: no order sent 2026-09-21  efb-2026-09-21-APH-S-83e7058edeaf
    BG       -6106.181934              0.0 DRY_RUN dry run: no order sent 2026-09-21   efb-2026-09-21-BG-S-9e9cb012a17d
   BNY       -5813.151377              0.0 DRY_RUN dry run: no order sent 2026-09-21  efb-2026-09-21-BNY-S-8a8a100b5ca7
  statuses: {'DRY_RUN': 27}


In [18]:
# And the same function checked against ids that ARE stored, in the
# 2026-09-25 log, which is the only stored log carrying them.
stored_ids = ROOT / 'live' / 'logs' / 'execution_2026-09-25.parquet'
if stored_ids.exists():
    frame = pd.read_parquet(stored_ids)
    recomputed = [alpaca.client_order_id(row.trade_date, str(row.ticker),
                                        'buy' if row.intended_notional >= 0 else 'sell')
                  for row in frame.itertuples(index=False)]
    matches = int(sum(a == b for a, b in zip(recomputed, frame['client_order_id'])))
    print('stored ids reproduced:', matches, 'of', len(frame))
    print(frame[['ticker', 'intended_notional', 'client_order_id']].head(3)
          .to_string(index=False))
    assert matches == len(frame), 'the client order id has drifted'

stored ids reproduced: 169 of 169
ticker  intended_notional                    client_order_id
   ABT        6409.835043  efb-2026-09-25-ABT-B-11738cdf1198
  ACGL        2102.155432 efb-2026-09-25-ACGL-B-ee304acbeaf1
   ADM       -6146.623411  efb-2026-09-25-ADM-S-f504d5023dfb


## 7. The cost estimate, split four ways

Four components, each in basis points of the paper NAV, and they must sum
to the total by construction. Only the **impact** is the cost of the trade
(it reads the change in weight, from flat on an establishment day). Spread,
commission and borrow scale with the book held, because they are the cost of
owning the position over the 21-session horizon.

In [19]:
from efb import costs as costs_mod

spread = costs_mod.spread_schedule(prices, DATA)
adv_map, _raw_adv, THIN = ej._adv_maps(KEPT_NAMES, DATA)
sigma = np.sqrt(np.maximum(SPECIFIC_KEPT, 1e-12))
spread_map = spread.reindex(KEPT_NAMES).fillna(spread.median()).to_numpy(float)
dollar_trade = np.abs(W_KEPT) * NAV
impact = (costs_mod.IMPACT_K * sigma
          * np.sqrt(np.maximum(dollar_trade, 0.0) / np.maximum(adv_map, 1.0)))
spread_bps = float(np.sum(spread_map * np.abs(W_KEPT))) * 1e4
commission_bps = float(np.sum(costs_mod.COMMISSION * np.abs(W_KEPT))) * 1e4
impact_bps = float(np.sum(impact * np.abs(W_KEPT))) * 1e4
short_gross = float(np.maximum(-W_KEPT, 0.0).sum())
borrow_bps = costs_mod.BORROW_RATE * short_gross * (ej.HORIZON / ej.TRADING_DAYS) * 1e4
print('INPUT  efb/costs.py: IMPACT_K', costs_mod.IMPACT_K, '| COMMISSION',
      costs_mod.COMMISSION, '| BORROW_RATE', costs_mod.BORROW_RATE)
print(f'  spread     {spread_bps:9.4f} bps')
print(f'  impact     {impact_bps:9.4f} bps')
print(f'  commission {commission_bps:9.4f} bps')
print(f'  borrow     {borrow_bps:9.4f} bps  on a short gross of {short_gross:.6f}')
print(f'  total      {spread_bps + impact_bps + commission_bps + borrow_bps:9.4f} bps')
print('  stored     ', PROPOSAL['cost_breakdown_bps'])
print('  thin ADV names (a median stands in for a measurement):', len(THIN))

INPUT  efb/costs.py: IMPACT_K 0.5 | COMMISSION 0.0001 | BORROW_RATE 0.02
  spread        5.1956 bps
  impact        0.4914 bps
  commission    1.0000 bps
  borrow        8.3333 bps  on a short gross of 0.500000
  total        15.0204 bps
  stored      {'spread': 5.195645358810164, 'impact': 0.4914133663273492, 'commission': 1.0, 'borrow': 8.333333333333341, 'total': 15.020392058470854}
  thin ADV names (a median stands in for a measurement): 0


## 8. The stored cross-check: what the 2026-09-25 run left behind

**Read this before quoting the section above.** The evening this notebook
traces is the newest close the artifacts in this checkout price. The
**2026-09-25** evening ran on a run tree that fetched sessions 2026-09-22 to
2026-09-25 from the vendor; those inputs were never committed, and
`build_proposal(as_of='2026-09-25')` refuses here with `no idio_momentum
signal row on 2026-09-25` because `models/XS-v1/specific_returns.parquet`
ends a session earlier. So the 09-25 book cannot be rebuilt in this
checkout. Two of its outputs were committed to the runtime tree and are read
below.

In [20]:
others = sorted((ROOT / 'live' / 'logs').glob('execution_*.parquet'))
print('stored execution logs:', [path.stem.replace('execution_', '') for path in others])
recon = pd.read_parquet(ROOT / 'live' / 'state' / 'reconciliation.parquet')
recon['trade_date'] = recon['trade_date'].astype(str).str.slice(0, 10)
cols = ['trade_date', 'gross', 'net', 'n_eff_kept', 'intended_notional',
        'expected_cost_bps', 'expected_spread_bps', 'expected_impact_bps',
        'expected_commission_bps', 'expected_borrow_bps']
stored = recon[cols].dropna(axis=1, how='all')
print()
print('OUTPUT live/state/reconciliation.parquet')
print(stored.to_string(index=False))
row = recon.loc[recon['trade_date'] == '2026-09-25']
if len(row):
    row = row.iloc[0]
    parts = (row['expected_spread_bps'] + row['expected_impact_bps']
             + row['expected_commission_bps'] + row['expected_borrow_bps'])
    print()
    print('the 09-25 establishment cost, four parts and their sum:')
    print(f'  spread {row["expected_spread_bps"]:.4f} + impact '
          f'{row["expected_impact_bps"]:.4f} + commission '
          f'{row["expected_commission_bps"]:.4f} + borrow '
          f'{row["expected_borrow_bps"]:.4f} = {parts:.4f} bps, against a stored '
          f'total of {row["expected_cost_bps"]:.4f} bps')
    assert abs(parts - row['expected_cost_bps']) < 0.01

stored execution logs: ['2026-09-03', '2026-09-21', '2026-09-22', '2026-09-24', '2026-09-25']

OUTPUT live/state/reconciliation.parquet
trade_date    gross           net  n_eff_kept  intended_notional  expected_cost_bps  expected_spread_bps  expected_impact_bps  expected_commission_bps  expected_borrow_bps
2026-09-03 1.000000 -1.471046e-15         NaN       1.000000e+05          75.343480                  NaN                  NaN                      NaN                  NaN
2026-09-21 0.968538 -9.992007e-16         NaN       2.733588e+05           3.964521                  NaN                  NaN                      NaN                  NaN
2026-09-24 0.822177  4.440892e-16   63.827664       1.000000e+06          67.337875                  NaN                  NaN                      NaN                  NaN
2026-09-25 0.900817  3.608225e-16   98.457145       1.000000e+06          15.094538             5.263688             0.497516                      1.0             8.333333

the

In [21]:
# And the stored 09-25 order log: 169 legs, an establishment day, gross
# intended notional equal to the whole paper NAV.
day_25 = ROOT / 'live' / 'logs' / 'execution_2026-09-25.parquet'
if day_25.exists():
    frame = pd.read_parquet(day_25)
    print('OUTPUT', day_25.relative_to(ROOT))
    print('  legs', len(frame), '| gross intended notional',
          float(frame['intended_notional'].abs().sum()))
    print('  statuses', frame['status'].value_counts().to_dict(),
          '| reason codes', frame['reason_code'].value_counts().to_dict())
    held = set(frame['ticker'])
    print('  LITE in it:', 'LITE' in held, '| MRNA in it:', 'MRNA' in held)
    print('  so the 169-name book and the book traced above are different days,')
    print('  and this notebook traces the one its inputs can price.')

OUTPUT live/logs/execution_2026-09-25.parquet
  legs 169 | gross intended notional 1000000.0
  statuses {'DRY_RUN': 169} | reason codes {'DRY_RUN': 169}
  LITE in it: False | MRNA in it: False
  so the 169-name book and the book traced above are different days,
  and this notebook traces the one its inputs can price.


## 9. What the email and the page show, and what each run_status field means

The email is sent after the proposal and the orders, never before, so a
failed send cannot block a run. Every run notifies, clean ones included: the
absence of the evening message is the alarm. The page is a JSON snapshot
under `docs/snapshot.schema.json`, which is the contract that pins the
builder and the TypeScript client together.

In [22]:
from live import notify, snapshot

SPECIAL = PROPOSAL['cost_breakdown_bps']
subject = notify.subject_text(status='ok', target_close=STAMP, dry_run=True,
                              orders=PROPOSAL['n_kept'],
                              worst_input=GATE['worst_input'],
                              worst_sessions_behind=GATE['worst_sessions_behind'],
                              inputs=GATE['inputs'])
body = notify.compose(
    status='ok', target_close=STAMP, dry_run=True, orders=PROPOSAL['n_kept'],
    gross=PROPOSAL['notional'], worst_input=GATE['worst_input'],
    worst_sessions_behind=GATE['worst_sessions_behind'],
    failures=GATE['failures'], inputs=GATE['inputs'],
    store='local parquet (dry run)', snapshot='off',
    cost_bps=SPECIAL['total'], cost_breakdown=SPECIAL,
    thin_adv=THIN, no_price=ej.universe_without_prices(DATA, as_of=CLOSE),
    cost_label='rebalance', brake_limit=NAV,
    positions_check={'note': 'dry run: the broker was not read'})
print('OUTPUT live/notify.py, the subject and the body')
print('subject:', subject)
print()
print(body)

OUTPUT live/notify.py, the subject and the body
subject: EFB ok 2026-09-21 | 169 proposed, none sent | stale 0

store: local parquet (dry run)
EFB live book 2026-09-21: ok, the run completed
Orders: dry run: 169 orders proposed, $1,000,000 gross, none sent
Staleness: worst input prices, 0 sessions behind (allowed 0).
Dropped for no price: BE, P.
Snapshot: off.
Day: rebalance. The absolute traded-notional brake of $1,000,000 applies, because the account already holds a book.
Cost: rebalance, 15.02 bps of NAV (spread 5.20 + impact 0.49 + commission 1.00 + borrow 8.33).
Positions: dry run: the broker was not read.


In [23]:
# The page. `snapshot.build` is the function the Cloudflare worker serves.
from live import snapshot

TRACED_BOOK = pd.DataFrame({'ticker': KEPT_NAMES, 'weight': W_KEPT})
CHOSEN = snapshot.chosen_row(PROPOSAL, root=DATA)
CONSTRUCTION_LABEL = snapshot.construction_label(PROPOSAL)
payload = snapshot.build(run={'target_close': STAMP, 'store': 'local',
                             'positions_check': {'note': 'dry run'}},
                        manifest=PROPOSAL, book=TRACED_BOOK,
                        construction=CHOSEN)
print('OUTPUT live/snapshot.py::build, the page payload')
print('construction:', payload['construction'])
print('book: n_names', payload['book']['n_names'], '| n_kept', payload['book']['n_kept'],
      '| gross', round(payload['book']['gross'], 6), '| cost bps',
      round(payload['book']['expected_cost_bps'], 4))
print('hedge:', {k: payload['hedge'][k] for k in ('idio_share_after_fmp',
                                                'max_abs_exposure_after_fmp')})
rows = [{'factor': k, 'before': v, 'after': payload['exposures_after_hedge'][k]}
        for k, v in payload['exposures_before_hedge'].items()]
print(pd.DataFrame(rows).head(8).to_string(index=False))

OUTPUT live/snapshot.py::build, the page payload
construction: min 20 shares (iterated to a fixed point)
book: n_names 169 | n_kept 169 | gross 1.0 | cost bps 15.0204
hedge: {'idio_share_after_fmp': 1.0, 'max_abs_exposure_after_fmp': 3.819861094100929e-15}
   factor    before         after
   market -0.137339 -9.089951e-16
     size  0.737781 -6.022960e-15
     beta  0.344132 -3.608225e-16
 momentum  0.969772 -7.268491e-16
 reversal  0.413121 -5.967449e-16
resid_vol  0.009538 -1.908196e-16
liquidity  0.594766  2.442491e-15
sector_10 -0.032686  7.806256e-18


In [24]:
# Every run_status field, and what it means.
row = staleness.run_status_row(GATE, run_date=STAMP, status='ok', dry_run=True,
                              n_orders=PROPOSAL['n_kept'],
                              gross_notional=PROPOSAL['notional'],
                              snapshot='off', cost_label='rebalance')
MEANING = {
    'target_close': 'the session the book was priced at, the key',
    'job': 'which cron wrote the row',
    'run_date': 'the day the job ran, which is not the close it priced',
    'status': 'ok, stale_stopped, error or market_closed; only ok marks the day done',
    'checked_at': 'when the gate ran',
    'max_input_staleness_days': 'calendar days to the oldest input, what the email quotes',
    'worst_input': 'the input furthest behind, in NYSE sessions',
    'worst_sessions_behind': 'how far behind it was, the number that decides',
    'n_inputs': 'how many inputs the gate looked at',
    'inputs': 'every input with its own date, source and allowance, as JSON',
    'failures': 'the inputs that failed their allowance, empty when clean',
    'detail': 'the error or refusal text, empty when clean',
    'notify_status': 'sent, failed or skipped: skipped means no channel was configured',
    'notify_failed': 'true when the message was not delivered, which exits nonzero',
    'n_orders': 'how many legs the morning proposed',
    'gross_notional': 'the book gross in dollars',
    'dry_run': 'true unless EFB_DRY_RUN is the literal false',
    'init': 'the run that seeded the store',
    'catch_up': 'the run closed more than one session at once',
    'catch_up_sessions': 'which sessions it closed',
    'splits': 'corporate actions applied, each named',
    'flags': 'names whose move tripped a sanity flag',
    'snapshot': 'the page write: on, off or the failure',
    'started_at': 'when the run began, used by the close gate',
    'cross_checks_capped': 'when the split cross-checks hit their cap',
    'establishment': 'the day the book was built from flat',
    'cost_label': 'establishment or rebalance, which cost the number is',
    'positions_check': 'the broker book against the store, counts and worst drift',
}
print('OUTPUT live/staleness.py::run_status_row')
for key, value in row.items():
    shown = str(value)
    shown = shown[:60] + '...' if len(shown) > 60 else shown
    print(f'  {key:26s} {shown:62s} {MEANING.get(key, "")}')
print()
print('and the state the page derives from the row:')
print(' ', staleness.run_state(row))

OUTPUT live/staleness.py::run_status_row
  target_close               2026-09-21                                                     the session the book was priced at, the key
  job                        live_daily                                                     which cron wrote the row
  run_date                   2026-09-21                                                     the day the job ran, which is not the close it priced
  status                     ok                                                             ok, stale_stopped, error or market_closed; only ok marks the day done
  checked_at                 2026-09-27T03:04:19.398216+00:00                               when the gate ran
  max_input_staleness_days   10                                                             calendar days to the oldest input, what the email quotes
  worst_input                None                                                           the input furthest behind, in NYSE sessions
  w

## 10. The clock, and the E11 criteria

The clock has **not started**. `live/clock.json` still holds the 2026-09-22 start
that was voided, because that loop ran on the frozen 2026-09-03 close and would
have produced thirty identical proposals. It restarts at the flip: `EFB_DRY_RUN`
goes false on the evening of Wednesday 2026-09-30, the orders fill on Thursday
2026-10-01, and that fill is day 1. The expected day 30 is read from the NYSE
calendar rather than counted in business days, which is why it is not 30
business days after the flip date.

The three criteria are copied verbatim from `sprints/E11/RESULTS.json`. **All
three are pending**: they are statements about thirty consecutive live trading
days, and they are evaluated with stored numbers once that window has closed.

In [25]:
results = json.loads((ROOT / 'sprints' / 'E11' / 'RESULTS.json').read_text())
clock = json.loads((ROOT / 'live' / 'clock.json').read_text())
FLIP = '2026-09-30'  # the owner's flip evening, when dry_run goes false
print('THE CLOCK')
print(f'  live/clock.json holds started={clock["started"]}, '
      f'day_1 {clock["day_1"]}, end_date {clock["end_date"]}')
for entry in clock.get('history', []):
    print(f'  voided start: day_1 {entry["day_1"]} ({entry["status"]})')
    print(f'    because {entry["reason"]}')
print(f'  RESULTS.json records day_1 {results["clock"]["day_1"]} and end '
      f'{results["clock"]["end_date"]}, the same voided start')
print('  status: NOT STARTED, starts at the flip.')
window = staleness.sessions(FLIP, '2027-06-30')
day_1 = next(stamp for stamp in window if stamp > pd.Timestamp(FLIP))
day_30 = window[window.index(day_1) + clock['trading_days'] - 1]
print(f'  the flip is the evening of {FLIP}: dry_run goes false, the orders fill')
print(f'  the next session, and that fill is day 1 = {day_1.date()}')
print(f'  expected day {clock["trading_days"]} from the NYSE calendar = {day_30.date()}')
print(f'  clock.start_clock("{day_1.date()}") is what that evening will write')
print('  RUN CONDITION: the book runs indefinitely; the thirty days are a')
print('  reporting window over the history, not the life of the run.')
print()
for name, block in results['criteria'].items():
    print(f'{name}: {block["verdict"].upper()}')
    print(f'  criterion: {block["criterion"]}')
    print(f'  stored numbers: {block["stored_numbers"] or "none yet, the window is open"}')
print()
print('what a failure would mean for a PM:')
print('  F11.1 a missed proposal is a day the book was not re-sized: the process,')
print('        not the signal, is what this criterion tests.')
print('  F11.2 an ex-ante risk number that does not reconcile is a page nobody can')
print('        act on, because the size of the book rests on it.')
print('  F11.3 realized vol outside the champion bias band means the book is not the')
print('        risk the research said it was, which is the whole premise of the loop.')

THE CLOCK
  live/clock.json holds started=True, day_1 2026-09-22, end_date 2026-11-02
  voided start: day_1 2026-09-22 (void)
    because static days: the loop ran on the frozen 2026-09-03 close and would produce thirty identical proposals
  RESULTS.json records day_1 2026-09-22 and end 2026-11-02, the same voided start
  status: NOT STARTED, starts at the flip.
  the flip is the evening of 2026-09-30: dry_run goes false, the orders fill
  the next session, and that fill is day 1 = 2026-10-01
  expected day 30 from the NYSE calendar = 2026-11-11
  clock.start_clock("2026-10-01") is what that evening will write
  RUN CONDITION: the book runs indefinitely; the thirty days are a
  reporting window over the history, not the life of the run.

F11.1: PENDING
  criterion: 30 consecutive trading days unattended with zero missed proposals.
  stored numbers: none yet, the window is open
F11.2: PENDING
  criterion: The ex-ante risk decomposition is stored every day and reconciles to the champion 

In [26]:
# Closing checklist: nothing typed by hand, and the traced name reached the end.
print('the traced name:', TRACE, '| in the book:', TRACE in STORED.index)
print('the cap case  :', CAP_CASE, '| capped to', round(float(work.loc[CAP_CASE, 'share_after']), 6))
print('closing checklist: clean')

the traced name: TER | in the book: True
the cap case  : MU | capped to 0.1
closing checklist: clean
